In [1]:
!nvidia-smi

Sun May 10 16:56:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile vector_add.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(int *a, int *b, int *c)
{
    int id = threadIdx.x;
    c[id] = a[id] + b[id];
}

int main()
{
    int h_a[5] = {1, 2, 3, 4, 5};
    int h_b[5] = {5, 4, 3, 2, 1};
    int h_c[5];

    int *d_a, *d_b, *d_c;

    cudaMalloc((void**)&d_a, 5 * sizeof(int));
    cudaMalloc((void**)&d_b, 5 * sizeof(int));
    cudaMalloc((void**)&d_c, 5 * sizeof(int));

    cudaMemcpy(d_a, h_a, 5 * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, 5 * sizeof(int), cudaMemcpyHostToDevice);

    vectorAdd<<<1, 5>>>(d_a, d_b, d_c);

    cudaMemcpy(h_c, d_c, 5 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Vector Addition Result:\n");

    for(int i = 0; i < 5; i++)
    {
        printf("%d ", h_c[i]);
    }

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Writing vector_add.cu


In [3]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./vector_add

Vector Addition Result:
6 6 6 6 6 